## Leetcode 197. Rising Temperature

In [0]:
from datetime import date
data = [(1,date(2015,1,1),10),(2,date(2015,1,2),25),
        (3,date(2015,1,3),20),(4,date(2015,1,4),30)]
Weather = spark.createDataFrame(data, ['id','recordDate','temperature'])
Weather.createOrReplaceTempView('Weather')
Weather.show()


Weather
| id | recordDate | temperature |
|----|------------|-------------|
| 1  | 2015-01-01 | 10          |
| 2  | 2015-01-02 | 25          |
| 3  | 2015-01-03 | 20          |
| 4  | 2015-01-04 | 30          |

### Spark SQL

In [0]:
#Method 1
spark.sql('''
          with final as (select id, recordDate, temperature,
lag(temperature) over (order by recordDate) as prev_temp,
lag(recordDate) over (order by recordDate) as prev_date
from Weather)

select id from final
where temperature > prev_temp 
and recordDate = prev_date + INTERVAL '1 day'
order by id

          ''').show()

In [0]:
#Method 2
spark.sql('''
  SELECT w2.id FROM Weather w1 JOIN Weather w2
  ON DATEDIFF(w2.recordDate, w1.recordDate) = 1
  WHERE w2.temperature > w1.temperature
''').show()


## Spark DataFrame API

In [0]:
from pyspark.sql.functions import lag, col, datediff
from pyspark.sql.window import Window
w = Window.orderBy(col('recordDate'))
Weather.withColumn('prev_temp', lag('temperature').over(w))\
    .withColumn('prev_date', lag('recordDate').over(w))\
    .filter((col('temperature') > col('prev_temp')) & (datediff(col('recordDate'), col('prev_date')) == 1))\
    .select('id').show()